#Uploading the data locally (monthly)

our data includes:
CRSP Monthly stock + variables:

ret (returns)

shrout (shares outstanding)

dlret (delisted returns)

vwretd (value-weighted returns)

years: 1975/01/01-2024/12/31

format: YYYY/MM/DD

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import statsmodels.formula.api as smf
from great_tables import GT, md, loc, style
import altair as alt
from IPython.display import display


In [ ]:
# Load proprietary CRSP and Compustat extracts (see README)
crsp_df = pd.read_csv("data/crsp_data.csv")
comp_df = pd.read_csv("data/comp_data.csv")

In [ ]:
# Read CSV and rename date column
fac = pd.read_csv('https://diether.org/prephd/08-factors.csv', parse_dates=['caldt'])
fac = fac.rename(columns={'caldt': 'date'})

# Define a function to compound daily returns
def compound_returns(x):
    return ((1 + x / 100).prod() - 1) * 100

# Dynamically create aggregation dictionary
agg_rules = {col: compound_returns for col in fac.columns if col not in ['date', 'rf']}
agg_rules['rf'] = 'last'

# Convert to monthly data using dynamically generated rules
fac_monthly = fac.set_index('date').resample('ME').agg(agg_rules).reset_index()

# Drop everything up until 1985-12-31 and reset index
fac_monthly = fac_monthly[fac_monthly['date'] >= '1972-1-1'].reset_index(drop=True)

fac_monthly

In [ ]:
crsp_df

In [ ]:
crsp_df["PRC"] = crsp_df["PRC"].abs()
crsp_df['mkt_cap'] = crsp_df['PRC']*crsp_df['SHROUT']
crsp_df['date'] = pd.to_datetime(crsp_df['date'])
crsp_df['match_month'] = crsp_df['date'].dt.to_period('M') - 1
crsp_df

In [ ]:
print(f'maximum CFACSHR: {crsp_df['CFACPR'].max()}')

In [ ]:
#matching months
comp_df['adate'] = pd.to_datetime(comp_df['adate']).dt.to_period('M')

comp_df

#

In [ ]:
#removing all the CFACSHR values that are zero so the adj_shrs won't be 0
crsp_df = crsp_df[crsp_df['CFACSHR'] > 0].copy()

#adjusting for stock splits and stuff
crsp_df['adj_shrs'] = crsp_df['SHROUT'] * crsp_df['CFACSHR']

# Sort to prevent misalignment in groupby
crsp_df = crsp_df.sort_values(by=['PERMNO', 'date'])

#Important ratio
# Calculate exact month we want to look for (36 months ago)
crsp_df['target_month'] = crsp_df['match_month'] - 36

# Create reference table of historical shares
historical_shares = crsp_df[['PERMNO', 'match_month', 'adj_shrs']].copy()
historical_shares = historical_shares.rename(
    columns={'match_month': 'merge_target', 'adj_shrs': 'shrs_t_36'}
)

# Merge historical shares back onto main dataframe
crsp_df = crsp_df.merge(
    historical_shares,
    left_on=['PERMNO', 'target_month'],
    right_on=['PERMNO', 'merge_target'],
    how='left'
)

# Calculate growth: If stock didn't exist 36 months ago it returns NaN
crsp_df['shrs_growth'] = crsp_df['adj_shrs'] / crsp_df['shrs_t_36']

# Clean up temporary columns
crsp_df = crsp_df.drop(columns=['target_month', 'merge_target'])


crsp_df

In [ ]:
# Ensure public_date is a datetime object
comp_df['public_date'] = pd.to_datetime(comp_df['public_date'])
comp_df = comp_df[comp_df['public_date'] >= '1972-1-1'].reset_index(drop=True)

# merge_asof requires both dataframes to be sorted by time
crsp_df = crsp_df.sort_values('date')
comp_df = comp_df.sort_values('public_date')

# Perform point-in-time merge
merged_df = pd.merge_asof(
    crsp_df,
    comp_df,
    left_on='date',           # The CRSP trading day
    right_on='public_date',   # The day financials became public
    left_by='PERMNO',         # Stock ID in CRSP
    right_by='permno',        # Stock ID in Compustat
    direction='backward'      # Only look into past
)

# Clean up rows that didn't have historical accounting data
merged_df = merged_df.dropna(subset=['bm', 'shrs_growth'])

# Creating portfolios
merged_df['bins'] = pd.qcut(
    merged_df['shrs_growth'],
    q=[0, 0.2, 0.6, 0.8, 1.0],
    labels=['0-20', '20-60', '60-80', '80-100']
)

print(merged_df[['PERMNO', 'date', 'public_date', 'shrs_growth', 'bins']].head(10))

#Crucial step: add dlret to avoid survivorship bias

In [ ]:
# Convert both to numeric, forcing errors to NaN, then fill NaN with 0
merged_df['RET'] = pd.to_numeric(merged_df['RET'], errors='coerce').fillna(0)
merged_df['DLRET'] = pd.to_numeric(merged_df['DLRET'], errors='coerce').fillna(0)

# Combine them: (1 + ret) * (1 + dlret) - 1
merged_df['RET'] = (1 + merged_df['RET']) * (1 + merged_df['DLRET']) - 1

In [ ]:
# Clean return column
merged_df['RET'] = pd.to_numeric(merged_df['RET'], errors='coerce').fillna(0)

#Spread
monthly_portfolios = merged_df.groupby(['date', 'bins'], observed=False)['RET'].mean().unstack().dropna()
monthly_portfolios['spread'] = monthly_portfolios['80-100'] - monthly_portfolios['0-20']
monthly_portfolios

In [ ]:
cum_returns = (1 + monthly_portfolios).cumprod()

# 2. Reset index to make 'date' a standard column for Altair
plot_df = cum_returns.reset_index()

# Ensure 'date' is a datetime object just in case
plot_df['date'] = pd.to_datetime(plot_df['date'])

# Define the order for the legend and color mapping
portfolios = ['0-20', '20-60', '60-80', '80-100', 'spread']

# 3. Create the main lines using transform_fold
performance_chart = alt.Chart(plot_df).transform_fold(
    fold=portfolios,
    as_=['Portfolio', 'Cumulative Value']
).mark_line().encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('Cumulative Value:Q', title='Cumulative Wealth (Growth of $1)'),

    # Map colors and set legend order
    color=alt.Color('Portfolio:N',
                    scale=alt.Scale(
                        domain=portfolios,
                        # Standard tab10 colors, with black for the spread
                        range=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', 'black']
                    )),

    # Make the spread dashed so it stands out
    strokeDash=alt.condition(
        alt.datum.Portfolio == 'spread',
        alt.value([5, 5]),  # Dashed line
        alt.value([0])      # Solid line
    ),

    # Make the spread thicker
    strokeWidth=alt.condition(
        alt.datum.Portfolio == 'spread',
        alt.value(2.5),
        alt.value(1.5)
    ),

    # Adjust opacity
    opacity=alt.condition(
        alt.datum.Portfolio == 'spread',
        alt.value(1.0),
        alt.value(0.8)
    )
)

# 4. Create a horizontal line at y=1 (the starting value of $1)
baseline = alt.Chart(pd.DataFrame({'y': [1]})).mark_rule(
    color='gray',
    strokeWidth=1,
    strokeDash=[2, 2]
).encode(
    y='y:Q'
)

# Combine layers and format the final chart
final_chart = (performance_chart + baseline).properties(
    title='Cumulative Portfolio Performance: Net Share Issuance',
    width=800,
    height=400
).interactive() # Enables zooming and panning

final_chart

In [ ]:
# Ensure index and columns are standard datetimes
monthly_portfolios.index = pd.to_datetime(monthly_portfolios.index)
fac_monthly['date'] = pd.to_datetime(fac_monthly['date'])

# Merge using portfolio index and factor 'date' column
merged_factors = pd.merge(
    monthly_portfolios,
    fac_monthly,
    left_index=True,
    right_on='date',
    how='inner'
)

# Set date back as index for regressions
merged_factors = merged_factors.set_index('date').dropna()

# Convert factors to decimal to match portfolio returns
merged_factors_dec = merged_factors.copy()
factor_cols = ['exmkt', 'smb', 'hml', 'umd', 'rf']
merged_factors_dec[factor_cols] = merged_factors_dec[factor_cols] / 100

print(merged_factors_dec.head())

In [ ]:
# Split dataset
merged_factors_dec.index = pd.to_datetime(merged_factors_dec.index)

# Create the splits
in_sample = merged_factors_dec.loc['1975-01-01':'1984-12-31'].copy()
out_of_sample = merged_factors_dec.loc['1985-01-01':].copy()

print(f"In-Sample months (1975-1984): {len(in_sample)}")
print(f"Out-of-Sample months (1985-Present): {len(out_of_sample)}")

In [ ]:
# FF4 and CAPM regressions

def run_regressions(df, period_name):
    ff4_results = {}
    capm_results = {}

    file_prefix = period_name.lower().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "")

    for col in ['0-20', '20-60', '60-80', '80-100', 'spread']:
        temp_df = df[[col, 'exmkt', 'smb', 'hml', 'umd', 'rf']].dropna().copy()

        # Zero-cost portfolio logic
        if col == 'spread':
            temp_df['excess_ret'] = temp_df[col]
        else:
            temp_df['excess_ret'] = temp_df[col] - temp_df['rf']

        # Run FF4
        ff4_model = smf.ols('excess_ret ~ exmkt + smb + hml + umd', data=temp_df).fit()
        ff4_results[col] = {
            'alpha': ff4_model.params['Intercept'],
            't_stat': ff4_model.tvalues['Intercept'],
            'beta_mkt': ff4_model.params['exmkt'],
            'beta_smb': ff4_model.params['smb'],
            'beta_hml': ff4_model.params['hml'],
            'beta_umd': ff4_model.params['umd'],
            'r_squared': ff4_model.rsquared
        }

        # Run CAPM
        capm_model = smf.ols('excess_ret ~ exmkt', data=temp_df).fit()
        capm_results[col] = {
            'alpha': capm_model.params['Intercept'],
            't_stat': capm_model.tvalues['Intercept'],
            'beta_mkt': capm_model.params['exmkt'],
            'r_squared': capm_model.rsquared
        }

    # Format FF4 Table
    ff4_df = pd.DataFrame(ff4_results).T.round(3).reset_index().rename(columns={'index': 'Portfolio'})
    ff4_table = (
        GT(ff4_df)
        .tab_header(title=f"Carhart 4-Factor: {period_name}")
        .cols_label(Portfolio="Bin", alpha="Alpha", t_stat="t-Stat", beta_mkt="Mkt Beta", beta_smb="SMB", beta_hml="HML", beta_umd="UMD", r_squared="R^2")
        .fmt_number(columns=["alpha", "beta_mkt", "beta_smb", "beta_hml", "beta_umd", "r_squared"], decimals=3)
        .fmt_number(columns=["t_stat"], decimals=2)
    )

    # Format CAPM Table
    capm_df = pd.DataFrame(capm_results).T.round(3).reset_index().rename(columns={'index': 'Portfolio'})
    capm_table = (
        GT(capm_df)
        .tab_header(title=f"CAPM: {period_name}")
        .cols_label(Portfolio="Bin", alpha="Alpha", t_stat="t-Stat", beta_mkt="Mkt Beta", r_squared="R^2")
        .fmt_number(columns=["alpha", "beta_mkt", "r_squared"], decimals=3)
        .fmt_number(columns=["t_stat"], decimals=2)
    )

    ff4_html = ff4_table.as_raw_html()
    capm_html = capm_table.as_raw_html()

    # Define filenames
    file_prefix = period_name.lower().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "")
    ff4_filename = f"{file_prefix}_ff4.html"
    capm_filename = f"{file_prefix}_capm.html"

    # Write to files and trigger automatic download
    for filename, html_content in [(ff4_filename, ff4_html), (capm_filename, capm_html)]:
        with open(filename, "w", encoding="utf-8") as f:
            f.write(html_content)
        files.download(filename)

    # Display the tables
    display(ff4_table)
    display(capm_table)

# Execute regressions for both periods
run_regressions(in_sample, "IN-SAMPLE (1975-1984)")
run_regressions(out_of_sample, "OUT-OF-SAMPLE (1985-Present)")

In [ ]:
# Cumulative charts
portfolios = ['0-20', '20-60', '60-80', '80-100', 'spread']

def build_chart(df, title_text, chart_type='wealth'):
    # Determine the math and formatting based on chart type
    if chart_type == 'wealth':
        cum_returns = (1 + df[['0-20', '20-60', '60-80', '80-100', 'spread']]).cumprod()
        y_title = 'Cumulative Wealth (Growth of $1)'
        baseline_y = 1
        y_axis_format = alt.Axis()
    else:
        cum_returns = (1 + df[['0-20', '20-60', '60-80', '80-100', 'spread']]).cumprod() - 1
        y_title = 'Cumulative Return (%)'
        baseline_y = 0
        y_axis_format = alt.Axis(format='%')

    plot_df = cum_returns.reset_index()

    lines = alt.Chart(plot_df).transform_fold(
        fold=portfolios, as_=['Portfolio', 'Value']
    ).mark_line().encode(
        x=alt.X('date:T', title='Date', axis=alt.Axis(format='%Y')),
        y=alt.Y(
            'Value:Q',
            title=y_title,
            scale=alt.Scale(zero=False),
            axis=y_axis_format
        ),
        color=alt.Color('Portfolio:N', scale=alt.Scale(domain=portfolios, range=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', 'black'])),
        strokeDash=alt.condition(alt.datum.Portfolio == 'spread', alt.value([5, 5]), alt.value([0])),
        strokeWidth=alt.condition(alt.datum.Portfolio == 'spread', alt.value(2.5), alt.value(1.5)),
        opacity=alt.condition(alt.datum.Portfolio == 'spread', alt.value(1.0), alt.value(0.8))
    )

    # Dynamic baseline
    baseline = alt.Chart(pd.DataFrame({'y': [baseline_y]})).mark_rule(
        color='gray', strokeWidth=1, strokeDash=[2, 2]
    ).encode(y='y:Q')

    return (lines + baseline).properties(
        title=title_text,
        width=800,
        height=400
    ).interactive()

# Generate Wealth Charts (Matches Ritter's Table 3)
wealth_in = build_chart(in_sample, 'In-Sample (1975-1984) - Wealth Index', chart_type='wealth')
wealth_out = build_chart(out_of_sample, 'Out-of-Sample (1985-Present) - Wealth Index', chart_type='wealth')

# Generate Return Charts
return_in = build_chart(in_sample, 'In-Sample (1975-1984) - Cumulative Return', chart_type='return')
return_out = build_chart(out_of_sample, 'Out-of-Sample (1985-Present) - Cumulative Return', chart_type='return')

# Display charts
display(wealth_in)
display(return_in)
display(wealth_out)
display(return_out)

In [ ]:
def generate_distribution_table(df, period_name):
    rolling_36m = (1 + df[['0-20', '80-100']]).rolling(window=36).apply(np.prod, raw=True) - 1
    rolling_36m = rolling_36m.dropna() * 100 # Convert to percentage

    # Define the percentiles to mimic Ritter's distribution rows
    percentiles = [0, 0.05, 0.25, 0.50, 0.75, 0.95, 1.0]
    labels = [
        'Lowest',
        '5th percentile',
        '25th percentile',
        'Median',
        '75th percentile',
        '95th percentile',
        'Highest'
    ]

    # Calculate the percentile values for both portfolios
    stats_80_100 = rolling_36m['80-100'].quantile(percentiles).tolist()
    stats_0_20 = rolling_36m['0-20'].quantile(percentiles).tolist()

    # Calculate and append the Mean
    stats_80_100.append(rolling_36m['80-100'].mean())
    stats_0_20.append(rolling_36m['0-20'].mean())
    labels.append('Mean')

    # Create the DataFrame
    dist_df = pd.DataFrame({
        'Rank': labels,
        'High Issuance (80-100)': stats_80_100,
        'Low Issuance (0-20)': stats_0_20
    })

    # Format to look exactly like Ritter's Table III
    gt_table = (
        GT(dist_df)
        .tab_header(
            title=f"Distribution of Three-Year Holding Period Returns",
            subtitle=f"For Net Share Issuance Portfolios in {period_name}"
        )
        .tab_spanner(
            label="Three-year holding period total return, in percent",
            columns=["High Issuance (80-100)", "Low Issuance (0-20)"]
        )
        .fmt_number(columns=["High Issuance (80-100)", "Low Issuance (0-20)"], decimals=2)
        .tab_style(
            style=style.borders(sides=["top", "bottom"], weight="2px"),
            locations=loc.body(rows=[len(labels)-1])
        )
    )

    return gt_table

table_iii_dist = generate_distribution_table(in_sample, "1975-1984")
display(table_iii_dist)
table_iii_dist_out = generate_distribution_table(out_of_sample, "1985-Present")
display(table_iii_dist_out)